## Loading Data

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/DeepLearning/MC/")

In [ ]:
!pip install stopwords

In [ ]:
import re
import nltk
import string
import stopwords
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder

import torch
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertForSequenceClassification, AdamW

In [ ]:
df = pd.read_csv("data/spam.csv", encoding="latin-1")

df = df.dropna(how="any", axis=1)
df.columns = ['target', 'message']
df['message_len'] = df['message'].apply(lambda x: len(x.split(' ')))

df.head()

## Data Analysis

In [ ]:
balance_counts = df.groupby('target')['target'].agg('count').values
print("Spam:", balance_counts[1], "Ham:", balance_counts[0])

In [ ]:
# Plot message length distribution with respect to class
plt.figure()
df[df.target == 'ham'].message_len.plot(bins=50, kind='hist', color='red', label='Ham messages', alpha=0.6)
df[df.target == 'spam'].message_len.plot(bins=10, kind='hist', color='blue', label='Spam messages', alpha=0.6)
plt.legend()
plt.xlabel("Message Length")
plt.show()

## Preprocessing

In [ ]:
stop_words = stopwords.get_stopwords(language='en')
stemmer = nltk.SnowballStemmer("english")
le = LabelEncoder()
le.fit(df['target'])

def clean_text(text):
    '''
    Make text lowercase, remove text in square brackets, links, punctuation and words containing numbers.
    '''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)

    return text

def preprocess_data(text):
    # Clean punctuation, urls, and so on
    text = clean_text(text)
    # Remove stop words
    text = ' '.join(word for word in text.split(' ') if word not in stop_words)
    # Stem all the words in the sentence
    text = ' '.join(stemmer.stem(word) for word in text.split(' '))

    return text

In [ ]:
df['message_clean'] = df['message'].apply(preprocess_data)
df['target_encoded'] = le.transform(df['target'])
df.head()

## Dataset

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, messages, targets, tokenizer, max_len):
        self.messages = messages
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.messages)

    def __getitem__(self, item):
        message = str(self.messages[item])
        target = self.targets[item]

        encoding = self.tokenizer.encode_plus(
            message,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt',
            truncation=True
        )

        return {
            'message_text': message,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(target, dtype=torch.long)
        }

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

df_train = df.sample(frac=0.8, random_state=42)
df_val = df.drop(df_train.index).reset_index(drop=True)
df_train = df_train.reset_index(drop=True)

def create_data_loader(df, tokenizer, max_len, batch_size):
    ds = SpamDataset(
        messages=df.message_clean.to_numpy(),
        targets=df.target_encoded.to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )

    return DataLoader(
        ds,
        batch_size=batch_size
    )

BATCH_SIZE = 32
MAX_LEN = 128

train_data_loader = create_data_loader(df_train, tokenizer, MAX_LEN, BATCH_SIZE)
val_data_loader = create_data_loader(df_val, tokenizer, MAX_LEN, BATCH_SIZE)

## Model

In [ ]:
# # DistilBERT model for text classification
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
model = model.to(device)

def train_epoch(model, data_loader, loss_fn, optimizer, device, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0

    for d in data_loader:
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        targets = d["targets"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        _, preds = torch.max(outputs[0], dim=1)
        loss = loss_fn(outputs[0], targets)

        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return correct_predictions.double() / n_examples, losses

def eval_model(model, data_loader, loss_fn, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            targets = d["targets"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            _, preds = torch.max(outputs[0], dim=1)
            loss = loss_fn(outputs[0], targets)

            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())

    return correct_predictions.double() / n_examples, losses

## Training

In [ ]:
def train_val(model, train_data_loader, val_data_loader, optimizer, loss_fn, device):
    # Wrap train_data_loader with tqdm for progress bar
    train_acc, train_loss = train_epoch(
        model,
        tqdm(train_data_loader, desc='Training', leave=False),
        loss_fn,
        optimizer,
        device,
        len(df_train)
    )

    print(f'Train loss: {np.mean(train_loss):.4f}, Train accuracy: {train_acc:.4f}')

    # Wrap val_data_loader with tqdm for progress bar
    val_acc, val_loss = eval_model(
        model,
        tqdm(val_data_loader, desc='Validation', leave=False),
        loss_fn,
        device,
        len(df_val)
    )

    print(f'Val loss: {np.mean(val_loss):.4f}, Val accuracy: {val_acc:.4f}')
    print()

    return train_acc, train_loss, val_acc, val_loss

In [ ]:
EPOCHS = 5
optimizer = AdamW(model.parameters(), lr=1e-6, correct_bias=False, no_deprecation_warning=True)

history = defaultdict(list)
best_accuracy = 0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')

    train_acc, train_loss, val_acc, val_loss = train_val(
        model,
        train_data_loader,
        val_data_loader,
        optimizer,
        torch.nn.CrossEntropyLoss(),
        device
    )

    for loss in train_loss:
        history['train_loss_iter'].append(loss)

    history['train_loss'].append(np.mean(train_loss))
    history['val_loss'].append(np.mean(val_loss))
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    if val_acc > best_accuracy:
        torch.save(model.state_dict(), 'model_weights.pth')
        best_accuracy = val_acc
        print("Model saved with validation accuracy:", best_accuracy.item())
        print()

In [ ]:
plt.plot(history['train_loss_iter'])

plt.title('Training loss at each iteration')
plt.ylabel('Loss')
plt.xlabel('Iteration')
plt.legend()
plt.show()

plt.plot(history['train_loss'], label='Train loss')
plt.plot(history['val_loss'], label='Validation loss')

plt.title('Training history')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.show()

## Cross Validation

In [ ]:
# Perform 3-fold cross validation
from sklearn.model_selection import KFold

kf = KFold(n_splits=3, shuffle=True, random_state=42)
accuracies = []
precisions = []
recalls = []
f1_scores = []
fold = 1
EPOCHS = 3

for train_index, test_index in kf.split(df):
    print(f"Fold {fold} of 3:")
    fold += 1
    df_train = df.iloc[train_index]
    df_val = df.iloc[test_index]

    train_data_loader = create_data_loader(df_train, tokenizer, MAX_LEN, BATCH_SIZE)
    val_data_loader = create_data_loader(df_val, tokenizer, MAX_LEN, BATCH_SIZE)

    model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
    model = model.to(device)

    optimizer = AdamW(model.parameters(), lr=2e-6, correct_bias=False, no_deprecation_warning=True)

    for epoch in range(EPOCHS):
        print(f'Epoch {epoch + 1}/{EPOCHS}')

        train_acc, train_loss, val_acc, val_loss = train_val(
            model,
            train_data_loader,
            val_data_loader,
            optimizer,
            torch.nn.CrossEntropyLoss(),
            device
        )

    accuracies.append(val_acc.item())

    preds = []
    targets = []

    with torch.no_grad():
        for d in val_data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            target = d["targets"].to(device)

            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            _, pred = torch.max(output[0], dim=1)

            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    preds = np.array(preds)
    targets = np.array(targets)

    tp = np.sum((preds == 1) & (targets == 1))
    tn = np.sum((preds == 0) & (targets == 0))
    fp = np.sum((preds == 1) & (targets == 0))
    fn = np.sum((preds == 0) & (targets == 1))

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1_score = 2 * (precision * recall) / (precision + recall)

    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1_score)

    print(f"Accuracy: {val_acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1_score:.4f}")
    print()

In [ ]:
# Print the result of 3-fold cross validation
print(f"Average accuracy: {np.mean(accuracies):.4f} in", accuracies)
print(f"Average precision: {np.mean(precisions):.4f} in", precisions)
print(f"Average recall: {np.mean(recalls):.4f} in", recalls)
print(f"Average F1 score: {np.mean(f1_scores):.4f} in", f1_scores)